# 06 — Online Loop Policy Comparison

**Author:** Jasjyot Singh

This notebook demonstrates the end-to-end online recommendation loop:

1. **Event Replay** — Replay historical interactions as a chronological stream
2. **State Warmup** — Build user/item/session state from replayed events
3. **Recommendation Serving** — Generate recommendations for sampled sessions
4. **Outcome Simulation** — Simulate user reactions to recommendations
5. **Logging** — Record exposure and outcome logs
6. **Offline Evaluation** — Compare policies on CTR, watch time, coverage, diversity, and creator spread

### System Architecture
```
Event Replay → State Update → Candidate Generation → Ranking → Serving → Logging → Simulation → Offline Evaluation
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Load Data

In [ ]:
from data.event_replay import load_interactions

master_df = load_interactions(os.path.abspath('../outputs/pipeline_sample.csv'))
print(f"Loaded {len(master_df):,} interactions")
print(f"Users: {master_df['user_id'].nunique()}, Items: {master_df['video_id'].nunique()}, Sessions: {master_df['session_id'].nunique()}")
master_df.head(3)

## 2. Event Replay — Stream Preview

Preview the first 10 events from the replay stream to verify the event schema mapping.

In [ ]:
from data.event_replay import replay_stream

csv_path = os.path.abspath('../outputs/pipeline_sample.csv')
sample_events = list(replay_stream(csv_path, max_events=10))

for i, evt in enumerate(sample_events[:5]):
    print(f"Event {i}: user={evt.user_id}, item={evt.item_id}, type={evt.event_type}, "
          f"watch={evt.watch_time:.0f}ms, creator={evt.creator_id}, cat={evt.category}")

## 3. State Warmup — Build User/Item/Session State

Process a subset of events to populate the in-memory state stores.

In [ ]:
from features.state_manager import StateManager
from data.event_schema import Event

state_mgr = StateManager()

# Warm up with first 5000 events
warmup_df = master_df.sort_values('time_ms').head(5000)
for _, row in warmup_df.iterrows():
    evt = Event(
        user_id=int(row['user_id']),
        item_id=int(row['video_id']),
        timestamp=int(row['time_ms']),
        event_type='watch' if row.get('play_time_ms', 0) > 0 else 'skip',
        watch_time=float(row.get('play_time_ms', 0)),
        creator_id=int(row['author_id']) if pd.notna(row.get('author_id')) else None,
        category=str(row['tag']) if pd.notna(row.get('tag')) else None,
        session_id=str(row['session_id']) if pd.notna(row.get('session_id')) else None,
    )
    state_mgr.process_event(evt)

print(f"State summary after warmup: {state_mgr.summary()}")

# Show a sample user state
sample_uid = list(state_mgr.user_states.keys())[0]
print(f"\nSample UserState (user={sample_uid}):")
print(state_mgr.get_user_state(sample_uid).to_dict())

## 4. Run Policy Backtest

Compare three ranking policies across the recommendation loop:
- **Popularity** — Pure engagement-score ranking
- **Recency Decay** — Freshness-boosted ranking
- **Hybrid** — Diversity-aware reranking

In [ ]:
from evaluation.policy_backtest import PolicyBacktest

backtest = PolicyBacktest(
    master_df=master_df,
    policies={
        'popularity': 'popularity',
        'recency_decay': 'freshness_boosted',
        'hybrid': 'diversity_aware',
    },
    n_users=50,
    n_sessions_per_user=3,
    k=20,
    warmup_events=5000,
    seed=42,
)

comparison_df = backtest.run()
comparison_df

## 5. Save Logs and Comparison

In [ ]:
# Save logs
log_dir = os.path.abspath('../outputs/logs')
os.makedirs(log_dir, exist_ok=True)
backtest.save_logs(log_dir)

# Save comparison
exp_dir = os.path.abspath('../outputs/experiments')
os.makedirs(exp_dir, exist_ok=True)

comparison_path = os.path.abspath('../outputs/policy_comparison.csv')
backtest.save_comparison(comparison_path)

# Also save to experiments dir
comparison_df.to_csv(os.path.join(exp_dir, 'policy_comparison.csv'), index=False)

print(f"\nOutputs saved:")
print(f"  Logs: {log_dir}")
print(f"  Comparison: {comparison_path}")

## 6. Policy Comparison — Visual Charts

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Online Loop Policy Comparison', fontsize=16, fontweight='bold')

metrics_to_plot = [
    ('ctr_proxy', 'CTR Proxy', 'CTR'),
    ('watch_time_proxy_ms', 'Watch Time Proxy (ms)', 'Watch Time (ms)'),
    ('catalog_coverage', 'Catalog Coverage', 'Coverage Ratio'),
    ('diversity', 'Category Diversity', 'Unique Categories / Session'),
    ('creator_spread', 'Creator Spread', 'Creator Ratio'),
]

colors = ['#4C72B0', '#DD8452', '#55A868']

for idx, (col, title, ylabel) in enumerate(metrics_to_plot):
    ax = axes[idx // 3][idx % 3]
    bars = ax.bar(comparison_df['policy'], comparison_df[col], color=colors[:len(comparison_df)])
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=15)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.4f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 5), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9)

# Summary text in the last subplot
ax_summary = axes[1][2]
ax_summary.axis('off')
summary_text = 'Sessions Evaluated\n'
for _, row in comparison_df.iterrows():
    summary_text += f"  {row['policy']}: {int(row['n_sessions_evaluated'])} sessions, {int(row['n_outcomes'])} outcomes\n"
ax_summary.text(0.1, 0.5, summary_text, transform=ax_summary.transAxes,
                fontsize=11, verticalalignment='center', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()

# Save the plot
plot_path = os.path.abspath('../outputs/experiments/policy_comparison_chart.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"Chart saved to {plot_path}")
plt.show()

## 7. Detailed Comparison Table

In [ ]:
display_cols = ['policy', 'ctr_proxy', 'watch_time_proxy_ms', 'catalog_coverage', 
                'diversity', 'creator_spread', 'n_exposures', 'n_outcomes']
available_cols = [c for c in display_cols if c in comparison_df.columns]
comparison_df[available_cols].style.format({
    'ctr_proxy': '{:.4f}',
    'watch_time_proxy_ms': '{:.0f}',
    'catalog_coverage': '{:.4f}',
    'diversity': '{:.2f}',
    'creator_spread': '{:.4f}',
}).set_caption('Online Loop Policy Comparison — Summary Metrics')

## Interpretation

The results above show how different ranking policies perform across the full recommendation loop:

- **CTR Proxy**: Higher values indicate the policy generates recommendations that users are more likely to interact with.
- **Watch Time Proxy**: Higher values mean users spend more time consuming recommended content.
- **Catalog Coverage**: Higher values indicate the system exposes users to a broader range of content.
- **Category Diversity**: Higher values mean each recommendation set contains more varied categories.
- **Creator Spread**: Higher values indicate impressions are distributed across more distinct creators.

This demonstrates the core tradeoff: **engagement-optimized policies (popularity)** tend to score higher on CTR/watch-time but lower on diversity/coverage, while **diversity-aware policies** sacrifice some engagement for broader, healthier content exposure.